# 🏆 CNN Ensemble — v3 + v6 (VGG+CBAM)
**Target: 76-78% | دمج أحسن موديلين عندنا**

### ⚠️ قبل التشغيل:
1. Runtime → Change runtime type → **T4 GPU** → Save
2. ارفع الملفين: `cnn_v3_final.h5` و `cnn_v6_final.h5`
3. اضغط **Run All** (Ctrl+F9)

In [ ]:
import tensorflow as tf
import numpy as np
import os, json, glob, shutil, random
import matplotlib.pyplot as plt
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

print('TF version:', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print('GPU:', gpus if gpus else '❌ NO GPU')
assert len(gpus) > 0, 'Enable GPU!'

SAVE_DIR      = '/content/models'
UNIFIED_VAL   = '/content/data/val'
os.makedirs(SAVE_DIR, exist_ok=True)

EMOTIONS    = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
NUM_CLASSES = 7
print('✅ Ready!')

In [ ]:
from google.colab import files

print('ارفع cnn_v3_final.h5 و cnn_v6_final.h5')
uploaded = files.upload()

for fname, data in uploaded.items():
    path = f'{SAVE_DIR}/{fname}'
    with open(path, 'wb') as f:
        f.write(data)
    print(f'✅ Saved: {path}')

In [ ]:
# v3 — CNN Residual من الصفر (grayscale 48x48)
model_v3 = tf.keras.models.load_model(f'{SAVE_DIR}/cnn_v3_final.h5')
print(f'✅ v3 loaded | params: {model_v3.count_params():,}')
print(f'   Input shape: {model_v3.input_shape}')

In [ ]:
# v6 — VGG16 + CBAM Attention (grayscale 48x48)
# محتاج نعرّف SpatialAttention قبل ما نحمّل الموديل
class SpatialAttention(layers.Layer):
    def __init__(self, kernel=7, **kwargs):
        super().__init__(**kwargs)
        self.conv = layers.Conv2D(1, kernel, padding='same', activation='sigmoid')

    def call(self, x):
        avg    = tf.reduce_mean(x, axis=-1, keepdims=True)
        mx     = tf.reduce_max(x,  axis=-1, keepdims=True)
        concat = tf.concat([avg, mx], axis=-1)
        scale  = self.conv(concat)
        return x * scale

    def get_config(self):
        return super().get_config()

model_v6 = tf.keras.models.load_model(
    f'{SAVE_DIR}/cnn_v6_final.h5',
    custom_objects={'SpatialAttention': SpatialAttention}
)
print(f'✅ v6 loaded | params: {model_v6.count_params():,}')
print(f'   Input shape: {model_v6.input_shape}')

In [ ]:
import subprocess

os.environ['KAGGLE_USERNAME'] = 'ahmedabolila'
os.environ['KAGGLE_KEY']      = '76db7068da271492790c5ea00702b28f'
!pip install -q kaggle

def download(dataset, dest):
    if os.path.exists(dest) and len(os.listdir(dest)) > 0:
        print(f'⏭️  {os.path.basename(dest)} already exists')
        return
    os.makedirs(dest, exist_ok=True)
    subprocess.run(['kaggle','datasets','download','-d',dataset,'-p',dest,'--unzip'], check=True)
    print(f'✅ {os.path.basename(dest)} downloaded')

download('jonathanoheix/face-expression-recognition-dataset', '/content/raw/ferplus')
download('shuvoalok/raf-db-dataset',                         '/content/raw/rafdb')
download('shawon10/ckplus',                                  '/content/raw/ckplus')

# Build val set
UNIFIED_TRAIN = '/content/data/train'
for split in [UNIFIED_TRAIN, UNIFIED_VAL]:
    for em in EMOTIONS:
        os.makedirs(f'{split}/{em}', exist_ok=True)

def copy_folder(src, emotion, split, prefix):
    dest = f'/content/data/{split}/{emotion}'
    imgs = []
    for ext in ['*.jpg','*.jpeg','*.png','*.bmp']:
        imgs += glob.glob(os.path.join(src, ext))
    for i, p in enumerate(imgs):
        dst = os.path.join(dest, f'{prefix}_{i:05d}.jpg')
        if not os.path.exists(dst):
            shutil.copy2(p, dst)

fer_map = {'angry':'angry','disgust':'disgust','fear':'fear',
           'happy':'happy','neutral':'neutral','sad':'sad','surprise':'surprise'}
for fer_em, our_em in fer_map.items():
    for src_split, dst_split in [('train','train'),('validation','val')]:
        for base in ['/content/raw/ferplus/images','/content/raw/ferplus']:
            src = os.path.join(base, src_split, fer_em)
            if os.path.exists(src):
                copy_folder(src, our_em, dst_split, f'fer_{fer_em}')
                break

raf_map = {'1':'surprise','2':'fear','3':'disgust',
           '4':'happy','5':'sad','6':'angry','7':'neutral'}
for raf_id, our_em in raf_map.items():
    for src_split, dst_split in [('train','train'),('test','val')]:
        for base in ['/content/raw/rafdb/DATASET','/content/raw/rafdb']:
            src = os.path.join(base, src_split, raf_id)
            if os.path.exists(src):
                copy_folder(src, our_em, dst_split, f'raf_{raf_id}')
                break

ck_map = {'anger':'angry','disgust':'disgust','fear':'fear',
          'happy':'happy','sadness':'sad','surprise':'surprise','contempt':'neutral'}
for ck_em, our_em in ck_map.items():
    for base in ['/content/raw/ckplus/train','/content/raw/ckplus',
                 '/content/raw/ckplus/CK+48']:
        src = os.path.join(base, ck_em)
        if os.path.exists(src):
            imgs = glob.glob(os.path.join(src,'**','*.png'), recursive=True)
            imgs += glob.glob(os.path.join(src,'**','*.jpg'), recursive=True)
            random.shuffle(imgs)
            cut = int(len(imgs) * 0.8)
            for i, p in enumerate(imgs[cut:]):
                dst = f'{UNIFIED_VAL}/{our_em}/ck_{ck_em}_{i:04d}.jpg'
                if not os.path.exists(dst): shutil.copy2(p, dst)
            break

total_val = sum(len(glob.glob(f'{UNIFIED_VAL}/{em}/*.jpg')) for em in EMOTIONS)
print(f'\n✅ Val set ready: {total_val:,} images')

In [ ]:
IMG_SIZE   = 48
BATCH_SIZE = 128
val_datagen = ImageDataGenerator(rescale=1./255)

# v3 predictions
print('Getting v3 predictions...')
gen_v3 = val_datagen.flow_from_directory(
    UNIFIED_VAL, target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode='categorical',
    color_mode='grayscale', shuffle=False
)
preds_v3  = model_v3.predict(gen_v3, verbose=1)
true_labels = gen_v3.classes
acc_v3 = np.mean(np.argmax(preds_v3, axis=1) == true_labels)
print(f'v3 accuracy: {acc_v3*100:.2f}%')

# v6 predictions
print('\nGetting v6 predictions...')
gen_v6 = val_datagen.flow_from_directory(
    UNIFIED_VAL, target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode='categorical',
    color_mode='grayscale', shuffle=False
)
preds_v6 = model_v6.predict(gen_v6, verbose=1)
acc_v6 = np.mean(np.argmax(preds_v6, axis=1) == true_labels)
print(f'v6 accuracy: {acc_v6*100:.2f}%')

In [ ]:
# جرب كل combination من الأوزان وشوف أنهي أحسن
print('Finding optimal ensemble weights...')
print('='*45)

best_acc    = 0
best_w_v3   = 0
best_w_v6   = 0
results     = []

for w_v3 in np.arange(0.0, 1.05, 0.05):
    w_v6  = 1.0 - w_v3
    ensemble = w_v3 * preds_v3 + w_v6 * preds_v6
    acc  = np.mean(np.argmax(ensemble, axis=1) == true_labels)
    results.append((w_v3, w_v6, acc))
    if acc > best_acc:
        best_acc  = acc
        best_w_v3 = w_v3
        best_w_v6 = w_v6

print(f'  v3 only  : {acc_v3*100:.2f}%')
print(f'  v6 only  : {acc_v6*100:.2f}%')
print(f'  Best mix : v3={best_w_v3:.2f} + v6={best_w_v6:.2f} → {best_acc*100:.2f}%')
print('='*45)

# Plot weights vs accuracy
ws   = [r[0] for r in results]
accs = [r[2]*100 for r in results]
plt.figure(figsize=(9, 4))
plt.plot(ws, accs, marker='o', color='steelblue')
plt.axvline(best_w_v3, color='red', linestyle='--', alpha=0.7,
            label=f'Best: v3={best_w_v3:.2f} ({best_acc*100:.2f}%)')
plt.xlabel('Weight for v3  (1-w = weight for v6)')
plt.ylabel('Accuracy (%)')
plt.title('Ensemble Weight Search')
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/ensemble_weights.png', dpi=150)
plt.show()

In [ ]:
# TTA Ensemble — أعلى دقة ممكنة
print('TTA Ensemble (5 passes each model)...')

tta_datagen = ImageDataGenerator(
    rescale=1./255, horizontal_flip=True,
    zoom_range=0.1, rotation_range=10
)

tta_v3, tta_v6 = [], []

for i in range(5):
    g = tta_datagen.flow_from_directory(
        UNIFIED_VAL, target_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH_SIZE, class_mode=None,
        color_mode='grayscale', shuffle=False
    )
    tta_v3.append(model_v3.predict(g, verbose=0))
    g = tta_datagen.flow_from_directory(
        UNIFIED_VAL, target_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH_SIZE, class_mode=None,
        color_mode='grayscale', shuffle=False
    )
    tta_v6.append(model_v6.predict(g, verbose=0))
    print(f'  Pass {i+1}/5 done')

avg_v3 = np.mean(tta_v3, axis=0)
avg_v6 = np.mean(tta_v6, axis=0)

# Apply best weights
tta_ensemble  = best_w_v3 * avg_v3 + best_w_v6 * avg_v6
tta_labels    = np.argmax(tta_ensemble, axis=1)
tta_acc       = np.mean(tta_labels == true_labels)

tta_v3_only   = np.mean(np.argmax(avg_v3, axis=1) == true_labels)
tta_v6_only   = np.mean(np.argmax(avg_v6, axis=1) == true_labels)

print('\n' + '='*55)
print(f'  v3 TTA only       : {tta_v3_only*100:.2f}%')
print(f'  v6 TTA only       : {tta_v6_only*100:.2f}%')
print(f'  Ensemble (no TTA) : {best_acc*100:.2f}%')
print(f'  Ensemble + TTA    : {tta_acc*100:.2f}%  🏆')
print(f'  v3 baseline       : 69.46%')
print(f'  Total improvement : +{(tta_acc - 0.6946)*100:.2f}%')
print('='*55)

In [ ]:
cm = confusion_matrix(true_labels, tta_labels)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=EMOTIONS, yticklabels=EMOTIONS)
plt.title(f'Ensemble v3+v6 — Confusion Matrix ({tta_acc*100:.2f}%)')
plt.ylabel('True'); plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/ensemble_confusion.png', dpi=150)
plt.show()
print(classification_report(true_labels, tta_labels, target_names=EMOTIONS))

In [ ]:
# Save ensemble config للاستخدام في المشروع
config = {
    'models': ['cnn_v3_final.h5', 'cnn_v6_final.h5'],
    'weights': [float(best_w_v3), float(best_w_v6)],
    'img_size': IMG_SIZE,
    'color_mode': 'grayscale',
    'emotions': EMOTIONS,
    'accuracy': {
        'v3_standard': float(acc_v3),
        'v6_standard': float(acc_v6),
        'ensemble_standard': float(best_acc),
        'ensemble_tta': float(tta_acc)
    }
}
with open(f'{SAVE_DIR}/ensemble_config.json', 'w') as f:
    json.dump(config, f, indent=2)

print('Ensemble Configuration:')
print(json.dumps(config, indent=2))

In [ ]:
from google.colab import files
print('Downloading...')
files.download(f'{SAVE_DIR}/ensemble_config.json')
files.download(f'{SAVE_DIR}/ensemble_weights.png')
files.download(f'{SAVE_DIR}/ensemble_confusion.png')
print('✅ Done!')